# Amazon ML Challenge 2026: Stage 0 Normalization (CPU)
### Shared preprocessing for Approach A and Track C
- Devanagari script transliteration (deterministic, zero external API)
- DBA marker splitting
- US State abbreviation <-> full name canonicalization
- Redacted house number detection ('##8' flagged as missing)
- Country-aware address parsing with generic France fallback
- Grouped 5-Fold CV split assignments


In [ ]:
# Environment & dependencies
import sys, os, re, unicodedata, gc
from pathlib import Path
import pandas as pd
import numpy as np

# Path resolution (Kaggle or local)
IS_KAGGLE = os.path.exists('/kaggle')
if IS_KAGGLE:
    INPUT_DIR = Path('/kaggle/input')
    # Locate dataset dir
    subdirs = list(INPUT_DIR.glob('*'))
    DATA_DIR = subdirs[0] if subdirs else Path('/kaggle/input/dataset')
    if (DATA_DIR / 'dataset').exists():
        DATA_DIR = DATA_DIR / 'dataset'
    elif (DATA_DIR / 'student_resource' / 'dataset').exists():
        DATA_DIR = DATA_DIR / 'student_resource' / 'dataset'
    WORK_DIR = Path('/kaggle/working/normalized')
else:
    DATA_DIR = Path('data/student_resource/dataset')
    WORK_DIR = Path('work/normalized')

WORK_DIR.mkdir(parents=True, exist_ok=True)
print(f"Data Dir: {DATA_DIR} | Work Dir: {WORK_DIR}")


In [ ]:
# Stage 0 Normalization Engine
DEVA_CONSONANTS = {
    '\u0915': 'k', '\u0916': 'kh', '\u0917': 'g', '\u0918': 'gh', '\u0919': 'ng',
    '\u091a': 'ch', '\u091b': 'chh', '\u091c': 'j', '\u091d': 'jh', '\u091e': 'ny',
    '\u091f': 't', '\u0920': 'th', '\u0921': 'd', '\u0922': 'dh', '\u0923': 'n',
    '\u0924': 't', '\u0925': 'th', '\u0926': 'd', '\u0927': 'dh', '\u0928': 'n',
    '\u092a': 'p', '\u092b': 'ph', '\u092c': 'b', '\u092d': 'bh', '\u092e': 'm',
    '\u092f': 'y', '\u0930': 'r', '\u0932': 'l', '\u0933': 'l', '\u0935': 'v',
    '\u0936': 'sh', '\u0937': 'sh', '\u0938': 's', '\u0939': 'h',
    '\u0958': 'q', '\u0959': 'kh', '\u095a': 'gh', '\u095b': 'z',
    '\u095c': 'd', '\u095d': 'dh', '\u095e': 'f', '\u095f': 'y'
}
DEVA_VOWELS = {
    '\u0905': 'a', '\u0906': 'aa', '\u0907': 'i', '\u0908': 'ee', '\u0909': 'u',
    '\u090a': 'oo', '\u090b': 'ri', '\u090e': 'e', '\u090f': 'e', '\u0910': 'ai',
    '\u0911': 'o', '\u0912': 'o', '\u0913': 'o', '\u0914': 'au'
}
DEVA_MATRAS = {
    '\u093e': 'a', '\u093f': 'i', '\u0940': 'ee', '\u0941': 'u', '\u0942': 'oo',
    '\u0943': 'ri', '\u0947': 'e', '\u0948': 'ai', '\u0949': 'o', '\u094a': 'o',
    '\u094b': 'o', '\u094c': 'au', '\u0946': 'e', '\u0945': 'e'
}
DEVA_VIRAMA = '\u094d'
DEVA_ANUSVARA = '\u0902'
DEVA_CANDRABINDU = '\u0901'
DEVA_VISARGA = '\u0903'
DEVA_NUKTA = '\u093c'
DEVA_REGEX = re.compile(r'[\u0900-\u097F]')

DEVA_WORD_MAP = {
    'प्राइवेट': 'private', 'लिमिटेड': 'limited', 'प्रा.': 'pvt', 'प्रा': 'pvt',
    'लि.': 'ltd', 'लि': 'ltd', 'एलएलपी': 'llp', 'कंपनी': 'company',
    'कॉर्पोरेशन': 'corporation', 'एंटरप्राइजेज': 'enterprises', 'इंडस्ट्रीज': 'industries',
    'सॉल्यूशंस': 'solutions', 'सर्विसेज': 'services', 'मार्केटिंग': 'marketing',
    'ट्रेडर्स': 'traders', 'प्रॉपर्टीज': 'properties', 'कंस्ट्रक्शंस': 'constructions',
    'टेक्नोलॉजीज': 'technologies', 'इन्वेस्टमेंट': 'investment', 'वेंचर्स': 'ventures',
    'डेवलपर्स': 'developers', 'इंटरनेशनल': 'international', 'महाराष्ट्र': 'maharashtra',
    'दिल्ली': 'delhi', 'कर्नाटक': 'karnataka', 'ಕರ್ನಾಟಕ': 'karnataka'
}

US_STATES = {
    "ALABAMA": "AL", "ALASKA": "AK", "ARIZONA": "AZ", "ARKANSAS": "AR",
    "CALIFORNIA": "CA", "COLORADO": "CO", "CONNECTICUT": "CT", "DELAWARE": "DE",
    "FLORIDA": "FL", "GEORGIA": "GA", "HAWAII": "HI", "IDAHO": "ID",
    "ILLINOIS": "IL", "INDIANA": "IN", "IOWA": "IA", "KANSAS": "KS",
    "KENTUCKY": "KY", "LOUISIANA": "LA", "MAINE": "ME", "MARYLAND": "MD",
    "MASSACHUSETTS": "MA", "MICHIGAN": "MI", "MINNESOTA": "MN", "MISSISSIPPI": "MS",
    "MISSOURI": "MO", "MONTANA": "MT", "NEBRASKA": "NE", "NEVADA": "NV",
    "NEW HAMPSHIRE": "NH", "NEW JERSEY": "NJ", "NEW MEXICO": "NM", "NEW YORK": "NY",
    "NORTH CAROLINA": "NC", "NORTH DAKOTA": "ND", "OHIO": "OH", "OKLAHOMA": "OK",
    "OREGON": "OR", "PENNSYLVANIA": "PA", "RHODE ISLAND": "RI", "SOUTH CAROLINA": "SC",
    "SOUTH DAKOTA": "SD", "TENNESSEE": "TN", "TEXAS": "TX", "UTAH": "UT",
    "VERMONT": "VT", "VIRGINIA": "VA", "WASHINGTON": "WA", "WEST VIRGINIA": "WV",
    "WISCONSIN": "WI", "WYOMING": "WY", "DISTRICT OF COLUMBIA": "DC"
}
US_STATE_CODES = set(US_STATES.values())
US_STATE_PATTERNS = [(re.compile(r'\b' + re.escape(k) + r'\b', re.I), v)
                     for k, v in sorted(US_STATES.items(), key=lambda x: -len(x[0]))]

LEGAL_REGEX = re.compile(
    r'\b(private limited|pvt ltd|pvt\. ltd\.|pvt|ltd|limited|llc|pllc|llp|inc|incorporated|'
    r'corp|corporation|co|company|sarl|sas|sci|sa|eurl|snc|gmbh|public limited)\b',
    re.I
)
DBA_REGEX = re.compile(r'\b(d/?b/?a|doing business as|trading as|t/?a)\b', re.I)
REDACTED_NUM_REGEX = re.compile(r'(^[#\*X]{2,}\d*|\b[#\*X]{2,}\d*\b)')

def transliterate_deva(text: str) -> str:
    if not text or not DEVA_REGEX.search(text):
        return text
    words = text.split()
    out = []
    for w in words:
        cw = w.strip('.,-–/()')
        if cw in DEVA_WORD_MAP:
            out.append(DEVA_WORD_MAP[cw])
            continue
        chars = list(w)
        res = []
        n = len(chars)
        for i in range(n):
            c = chars[i]
            if c in DEVA_CONSONANTS:
                res.append(DEVA_CONSONANTS[c])
                next_c = chars[i+1] if i + 1 < n else None
                if next_c is None or (next_c not in DEVA_MATRAS and next_c != DEVA_VIRAMA):
                    if i + 1 < n and (chars[i+1] in DEVA_CONSONANTS or chars[i+1] in DEVA_VOWELS):
                        res.append('a')
            elif c in DEVA_VOWELS:
                res.append(DEVA_VOWELS[c])
            elif c in DEVA_MATRAS:
                res.append(DEVA_MATRAS[c])
            elif c in (DEVA_ANUSVARA, DEVA_CANDRABINDU):
                res.append('n')
            elif c == DEVA_VISARGA:
                res.append('h')
            elif c not in (DEVA_VIRAMA, DEVA_NUKTA):
                res.append(c)
        out.append(''.join(res))
    return ' '.join(out)

def strip_accents(text: str) -> str:
    if not text: return ""
    return ''.join(c for c in unicodedata.normalize('NFKD', text) if not unicodedata.combining(c))

def clean_basic(text: str) -> str:
    if not text or text == "nan": return ""
    text = strip_accents(text).lower().replace('&', ' and ')
    text = re.sub(r'[^\w\s]', ' ', text)
    return re.sub(r'\s+', ' ', text).strip()

def normalize_record_py(eid, name, addr, country):
    name_str = str(name).strip() if pd.notna(name) else ""
    addr_str = str(addr).strip() if pd.notna(addr) else ""
    
    # Name
    translit_name = transliterate_deva(name_str)
    dba_m = DBA_REGEX.search(translit_name)
    primary = translit_name[:dba_m.start()].strip() if dba_m else translit_name
    name_core = clean_basic(primary)
    name_core = LEGAL_REGEX.sub('', name_core)
    name_core = re.sub(r'\s+', ' ', name_core).strip()
    
    # Address
    translit_addr = transliterate_deva(addr_str)
    is_redacted = "1" if REDACTED_NUM_REGEX.search(translit_addr) else "0"
    addr_clean = REDACTED_NUM_REGEX.sub(' ', translit_addr) if is_redacted == "1" else translit_addr
    
    # House number
    house_num = ""
    if is_redacted == "0":
        dm = re.search(r'\b(?:door\s*no\.?|h\.?\s*no\.?|plot\s*no\.?|kh\s*no\.?|unit\s*no\.?|no\.)\s*([0-9]+[a-zA-Z0-9/\-]*)\b', addr_clean, re.I)
        if dm:
            house_num = dm.group(1).lower()
        else:
            lm = re.search(r'^\s*([0-9]+[a-zA-Z]?)\b', addr_clean)
            if lm:
                house_num = lm.group(1).lower()
            else:
                fm = re.search(r'\b([0-9]+[a-zA-Z]?)\b', addr_clean[:35])
                if fm: house_num = fm.group(1).lower()
                
    # State canonicalization
    state_code = ""
    for pat, code in US_STATE_PATTERNS:
        if pat.search(addr_clean):
            state_code = code
            addr_clean = pat.sub(f' {code} ', addr_clean)
            break
    if not state_code:
        for t in re.findall(r'\b[A-Za-z]{2}\b', addr_clean):
            if t.upper() in US_STATE_CODES:
                state_code = t.upper()
                break
                
    addr_core = clean_basic(addr_clean)
    addr_core = re.sub(r'\b(rd)\b', 'road', addr_core)
    addr_core = re.sub(r'\b(st)\b', 'street', addr_core)
    addr_core = re.sub(r'\b(ave|av)\b', 'avenue', addr_core)
    addr_core = re.sub(r'\b(blvd|bd)\b', 'boulevard', addr_core)
    addr_core = re.sub(r'\s+', ' ', addr_core).strip()
    
    return {
        "entity_id": str(eid).strip(),
        "name_core": name_core,
        "name_translit": translit_name,
        "addr_core": addr_core,
        "house_number": house_num,
        "is_house_redacted": is_redacted,
        "state_code": state_code,
        "country": country if pd.notna(country) else "UNKNOWN"
    }


In [ ]:
# Stream process and convert all files to Parquet
files_to_process = [
    ('train/train_source1.tsv', 'train_s1_norm.parquet'),
    ('train/train_source2.tsv', 'train_s2_norm.parquet'),
    ('train/train_source3.tsv', 'train_s3_norm.parquet'),
    ('test/test_source1.tsv', 'test_s1_norm.parquet'),
    ('test/test_source2.tsv', 'test_s2_norm.parquet'),
    ('test/test_source3.tsv', 'test_s3_norm.parquet'),
]

for src_rel, dst_name in files_to_process:
    src_path = DATA_DIR / src_rel
    dst_path = WORK_DIR / dst_name
    if not src_path.exists():
        print(f"Skipping {src_rel} (not found)")
        continue
    print(f"Processing {src_rel} -> {dst_name}...")
    chunk_list = []
    for chunk in pd.read_csv(src_path, sep='\t', chunksize=200000):
        rows = [normalize_record_py(r['entity_id'], r['business_name'], r['business_address'], r['country'])
                for _, r in chunk.iterrows()]
        chunk_list.append(pd.DataFrame(rows))
        gc.collect()
    df_out = pd.concat(chunk_list, ignore_index=True)
    df_out.to_parquet(dst_path, index=False)
    print(f"  Successfully wrote {len(df_out):,} records to {dst_path}")
    del chunk_list, df_out
    gc.collect()

print("Stage 0 Normalization Complete!")
